# TissueMC: why do R3's warps wait so long?

R3 raised the active lanes per warp from 4.74 to 21.61 of 32 but ran about 6% slower than R1, because each warp waited about 98 cycles between issued instructions instead of 20. The counters collected so far do not say why.

This notebook collects **every** Nsight Compute counter for R1 and R3 on the same workload and prints the ones that bear on the question side by side: the warp stall reasons, the instruction counts, the local-memory traffic and the launch and occupancy figures.

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Nothing else to record.

**Step 1 is a 1-minute check** of the whole path on a tiny run. If it prints an error, **stop there and paste the output**; do not run the long steps. Otherwise *Runtime → Run all*; the two full profiles replay the kernel about forty times each, roughly 3 to 6 minutes apiece. It ends with **one digest** for you to paste back. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
!git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git
%cd tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

In [ ]:
# 1. Quick check of the whole path on a tiny run: collect, parse, digest. Also shows the real CSV
# header, which tells us the column layout Nsight Compute actually writes.
!tissuemc-profile --rung 1 --full --photons 20000 --binary build/tissuemc --output results/smoke-full.json
!head -c 1200 results/smoke-full.ncu.csv
!tissuemc-profile --digest results/smoke-full.json results/smoke-full.json

In [ ]:
# 2. Every counter for R1 (500,000 photons).
!tissuemc-profile --rung 1 --full --photons 500000 --binary build/tissuemc --output results/gpu-r1-full.json

In [ ]:
# 3. Every counter for R3, the same workload.
!tissuemc-profile --rung 3 --full --photons 500000 --binary build/tissuemc --output results/gpu-r3-full.json

In [ ]:
# 4. The digest: A is R1, B is R3. Paste this cell's output back.
!tissuemc-profile --digest results/gpu-r1-full.json results/gpu-r3-full.json